In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

from collections import defaultdict
from pathlib import Path
import pandas as pd

Mounted at /content/gdrive


In [ ]:
!cp gdrive/MyDrive/LAB1_TMS/extras/preprocessing_functions.py preprocessing_functions.py
!cp gdrive/MyDrive/LAB1_TMS/extras/corpus.zip corpus.zip

# Loading the data

In [ ]:
import nltk
from preprocessing_functions import load_president_speeches, remove_punctuation, remove_numbers
data = load_president_speeches()
data.head(2)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


,year,president,text
0,1957,Eisenhower,"The Price of Peace\nMr. Chairman, Mr. Vice Pre..."
1,1969,Nixon,"Senator Dirksen, Mr. Chief Justice, Mr. Vice P..."


In [ ]:
#Pre-process
data["preprocessed"] = data.text.str.lower()
data.preprocessed = data.preprocessed.apply(remove_numbers)
data.preprocessed =  data.preprocessed.apply(remove_punctuation)
data["tokens"] = data.preprocessed.str.split()
data.head(3)
print(len(data))

56


In [ ]:
data.head(2)

,year,president,text,preprocessed,tokens
0,1973,Nixon,"Mr. Vice President, Mr. Speaker, Mr. Chief Jus...",mr vice president mr speaker mr chief jus...,"[mr, vice, president, mr, speaker, mr, chief, ..."
1,1833,Jackson,"Fellow citizens, the will of the American peop...",fellow citizens the will of the american peop...,"[fellow, citizens, the, will, of, the, america..."


# Exercise 0: Simple Document Representation

Given the query: "senator mathias", write code that returns all the documents that contain it. Hints:

1.   Use the preprocessed column of the dataframe and the str.contains function\
TRY: https://www.geeksforgeeks.org/python-pandas-series-str-find/
2.   Create a new column in the dataframe that contains the True or False value.


In [ ]:
# EXERCISE: Your code here
data['search'] = data['preprocessed'].
display(data[data.search])
print(data[data.search].shape)

,year,president,text,preprocessed,tokens,search
18,1985,Reagan,"Senator Mathias, Chief Justice Burger, Vice Pr...",senator mathias chief justice burger vice pr...,"[senator, mathias, chief, justice, burger, vic...",True


(1, 6)


Given the query: "senator mathias", write code that returns all the documents ranked based on the tf-idf similarity. Hints:

1.  Use the tf-idf vectorizer from the sklearn
2.  Fit the documents in the vectorizer
3.  Write a function that takes the document vectors, a query (a text) and the vectorizer and returns the cosine similarity between the 2 vectors.
4.  Display the president and the year of the documents that have similarity with the query > 0.02. The query is: "senator mathias"

In [ ]:
# Your code here

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
docs_tfidf = vectorizer.fit_transform(data.preprocessed)

In [ ]:
def get_tf_idf_query_similarity(vectorizer, docs_tfidf, query):
    query_tfidf = vectorizer.transform([query])
    similarities = cosine_similarity(query_tfidf, docs_tfidf).flatten()
    return similarities

In [ ]:
ex_query = 'senator mathias'
similarities = get_tf_idf_query_similarity(vectorizer, docs_tfidf, ex_query)

display(similarities)

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.01507705, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.01437518, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.00910075, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.00754142,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.05175934, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        ])

In [ ]:
for doc_sim in similarities:
  # EXERCISE: Display the documents that have similarity with the query > 0.02
  if doc_sim > 0.02:
      display(data[['president', 'year']].iloc[list(similarities).index(doc_sim)])

president    Reagan
year         1985  
Name: 38, dtype: object

# Exercise 1: Static Word Embeddings

---




Steps:

Part 1 **Training Word2vec on a dataset**:
- Train a Word2Vec model on the presidents' speeches. You must pass to Word2Vec a list of lists of tokens (a list of tokenized sentences). Use CBOW and a window of 5. Documentation: https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec
- Find the words most similar to, e.g. "state", "states", "president", or others.
- Compare the effectiveness of CBOW and Skipgram. Which one works better?


Part 2 **Using pre-trained static models**:
- Load a pre-trained gensim model (available models: https://github.com/RaRe-Technologies/gensim-data#Models) (Choose a small one to avoid waiting for the download, e.g. `glove-wiki-gigaword-100`).
- Evaluate the following relations:
  - king + woman - man = ?
  - France - Paris + London = ?
  - Find out some for yourselves

**Question: Find the words most similar to "italy" using the 2 models (trained and pretrained). What do you observe? Why? Any solution?**



In [ ]:
# Your code here

from gensim import downloader
from gensim.models import Word2Vec

In [ ]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
data['sent_tokens'] = data.text.str.lower()

data.sent_tokens = data.sent_tokens.str.replace(r'\d+', ' ', regex=True)
data.sent_tokens = data.sent_tokens.apply(lambda input_text : [remove_punctuation(t).split() for t in nltk.sent_tokenize(input_text)])

!

In [ ]:
len(data.sent_tokens.sum())

4854

In [ ]:
len(data.sent_tokens)

56

In [ ]:
corpus = data.sent_tokens.sum() # set a list of sentences rather than list of list of words
# now use the slides code for training!

model = Word2Vec(sentences=corpus, size=100, window=5, sg=1, seed=42)
# use sg=1 for skip-gram and sg=0 for CBoW

# set similar words to:
# states, war, state
print(model.wv.most_similar('states', topn=10))

# try also italy and see what happens!

[('powers', 0.912544310092926), ('defend', 0.8784132599830627), ('united', 0.8738170266151428), ('constitution', 0.8664664626121521), ('protect', 0.8534152507781982), ('federal', 0.8480732440948486), ('reserved', 0.8475136756896973), ('authority', 0.8374835252761841), ('several', 0.8319411277770996), ('laws', 0.8169540762901306)]


In [ ]:
import numpy as np
try:
    italy_vector = model.wv.most_similar('italy', topn=10)
except KeyError as e:
    italy_vector = np.zeros(100)

print(italy_vector)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]


In [ ]:
downloader.info()['models'].keys()

dict_keys(['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis'])

In [ ]:
wiki_model = downloader.load('glove-wiki-gigaword-100')

[==================================================] 100.0% 128.1/128.1MB downloaded


In [ ]:
wiki_model.wv['president']

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: DeprecationWarning: Call to deprecated `wv` (Attribute will be removed in 4.0.0, use self instead).
  """Entry point for launching an IPython kernel.


array([-0.064549, -0.13812 ,  0.50017 ,  0.41434 ,  0.45832 , -0.048331,
       -0.21651 ,  0.34987 , -0.83235 , -0.62282 , -0.40099 , -0.31978 ,
        0.42928 , -0.026035, -0.16171 , -0.32513 ,  0.77604 , -0.39852 ,
       -0.67013 , -0.55876 , -0.39863 , -0.28143 ,  0.68614 ,  0.29229 ,
       -0.7146  ,  0.36995 , -0.45309 , -0.23193 ,  0.76916 , -0.031589,
        1.0603  ,  1.5446  , -0.13272 ,  0.33701 , -0.97561 ,  0.73617 ,
        0.60841 ,  0.558   , -0.39274 ,  0.1121  , -1.0474  , -0.03692 ,
        0.99908 , -0.044925, -0.41186 ,  0.19605 ,  0.093281, -0.59623 ,
       -0.60765 , -0.4742  , -0.30162 , -0.70128 , -0.20055 ,  0.99519 ,
        0.20088 , -2.5636  , -0.14627 ,  0.77824 ,  1.4913  ,  0.29724 ,
       -0.12084 , -0.060082,  0.079881, -0.38348 ,  0.41371 , -0.41284 ,
        0.61702 ,  1.1316  , -0.047434,  0.2848  ,  0.57694 , -0.45501 ,
       -0.76359 , -1.1659  ,  0.068946, -0.22498 ,  0.18691 ,  0.64142 ,
       -1.7505  , -0.091248,  0.85848 , -0.52203 , 

In [ ]:
wiki_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)

[('queen', 0.7698541283607483),
 ('monarch', 0.6843380928039551),
 ('throne', 0.6755735874176025),
 ('daughter', 0.6594556570053101),
 ('princess', 0.6520534753799438)]

In [ ]:
wiki_model.most_similar(positive=['france', 'london'], negative=['paris'], topn=5)

[('britain', 0.8921419978141785),
 ('england', 0.8002905249595642),
 ('australia', 0.7940487861633301),
 ('scotland', 0.7838967442512512),
 ('british', 0.7608305215835571)]

# Exercise 2: Embeddings with multiword ngrams

---



**Training Word2vec on a dataset**:
- Train a bigram detector using gensim.
- Train a Word2Vec model on the presidents' speeches. You must pass to Word2Vec a list of lists of bigram tokens (a list of tokenized sentences).
- Find the words most similar to, e.g. "united_states", "american_people" or others.
- Try both word2vec architectures.

**Have you seen the sparsity problem?**

In [ ]:
from gensim.models import Phrases

bigram_generator = Phrases(data.tokens)

In [ ]:
bigram_tokens = bigram_generator[data.tokens]

/usr/local/lib/python3.7/dist-packages/gensim/models/phrases.py:598: UserWarning: For a faster implementation, use the gensim.models.phrases.Phraser class
  warnings.warn("For a faster implementation, use the gensim.models.phrases.Phraser class")


In [ ]:
# use bigram_tokens as your new corpus (same code as before!)
model = Word2Vec(sentences=bigram_tokens, size=100, window=5, sg=1, seed=42)

In [ ]:
print(model.wv.most_similar('united_states', topn=10))

[('constitution', 0.9667254090309143), ('states', 0.9621188044548035), ('whole', 0.9528996348381042), ('rights', 0.9472708106040955), ('powers', 0.9415111541748047), ('executive', 0.9414318799972534), ('laws', 0.9409316778182983), ('general_government', 0.9398400187492371), ('interests', 0.9381714463233948), ('union', 0.9362156391143799)]


In [ ]:
print(model.wv.most_similar('american_people', topn=10))

[('second', 0.9975712299346924), ('destinies', 0.9974365234375), ('encouragement', 0.9973673224449158), ('officer', 0.9972285032272339), ('assume', 0.9971164464950562), ('expressed', 0.9969063401222229), ('same', 0.9968450665473938), ('source', 0.9966413974761963), ('direction', 0.9964177012443542), ('lot', 0.996334433555603)]


# Exercise 3

You can represent a document as the weighted sum of the word embeddings of the terms it contains, where the term weights are given by, e.g. Tf-Idf.

- Compute the document embeddings for the US presidential speeches corpus.
- Write a function that, given a document (speech) id, finds the top n most similar documents (adapt the previous one for most similar terms).


Hints (but there may be other / better solutions):

- To obtain the TfIdf values from the 'tfidf' matrix returned by fit_transform, use 'tfidf[doc_id, term_index]'. The list of terms (features) is returned by 'vectorizer.get_feature_names_out()'
- To obtain the Word2Vec embedding for a word, use 'model.wv[word]', where 'model = Word2Vec(...)'
- To obtain the document embedding, iterate over all of its words (that are present both as features for the TfIdf vectorizer and in the Word2Vec model vocabulary), and sum over all the products: (model.wv[word] * tfidf[doc_id, word_index])

In [ ]:
# Your code here

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
docs_tfidf = vectorizer.fit_transform(data.preprocessed)

model = Word2Vec(sentences=corpus, window=5, sg=1, seed=42) # we use skipgram

In [ ]:
tf_idf_names = vectorizer.get_feature_names_out()

In [ ]:
skip_gram_words = model.wv.vocab

In [ ]:
tfidf_feature_to_index = {word: i for i, word in enumerate(tf_idf_names)}

def tfidf_weight(doc_id, term):
    return docs_tfidf[doc_id, tfidf_feature_to_index[term]]

In [ ]:
skip_gram_set = set(skip_gram_words)
tfidf_set = set(tf_idf_names)

common_terms = skip_gram_set.intersection(tfidf_set)

In [ ]:
def get_document_embedding(doc_id, common_terms):
    document = data.tokens[doc_id]
    embedding = np.zeros_like(model.vector_size)
    for word in document:
        if word in common_terms:
            embedding = embedding + model.wv[word] * tfidf_weight(doc_id, word)

    return embedding

In [ ]:
doc_embeddings = np.zeros((data.shape[0], model.vector_size))

for i in range(data.shape[0]):
    doc_embeddings[i] = get_document_embedding(i, common_terms)

In [ ]:
doc_embeddings.shape

(56, 100)

In [ ]:
doc_cos = cosine_similarity(doc_embeddings, doc_embeddings)

def top_n_most_similar_document(doc_id, doc_sim, top_n=5):
    current_doc_sim = doc_sim[doc_id]
    return data[['president', 'year', 'text']].loc[(-current_doc_sim).argsort()[1:(top_n+1)]]

In [ ]:
doc_cos.shape

(56, 56)

In [ ]:
data.iloc[2,:].text

'Fellow citizens, without solicitation on my part, I have been chosen by the free and voluntary suffrages of my countrymen to the most honorable and most responsible office on earth. I am deeply impressed with gratitude for the confidence reposed in me. Honored with this distinguished consideration at an earlier period of life than any of my predecessors, I can not disguise the diffidence with which I am about to enter on the discharge of my official duties.\n\nIf the more aged and experienced men who have filled the office of President of the United States even in the infancy of the Republic distrusted their ability to discharge the duties of that exalted station, what ought not to be the apprehensions of one so much younger and less endowed now that our domain extends from ocean to ocean, that our people have so greatly increased in numbers, and at a time when so great diversity of opinion prevails in regard to the principles and policy which should characterize the administration of

In [ ]:
top_n_most_similar_document(2, doc_cos, 5)

president  year  \
5   Jackson   1833   
1   Harrison  1889   
47  Taft      1909   
49  McKinley  1897   
41  Hayes     1877   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

# Exercise 4

Use Doc2Vec (https://radimrehurek.com/gensim/models/doc2vec.html) to compute the embeddings for the documents in the presidential speeches dataset. Write a function that finds the most similar documents for a given document(adapt the previous function).

In [ ]:
# Your code here

from gensim.models.doc2vec import Doc2Vec, TaggedDocument

In [ ]:
documents = [TaggedDocument(doc, [i]) for i, doc in enumerate(data.tokens)]

In [ ]:
doc_model = Doc2Vec(documents, window=5, seed=42)

In [ ]:
# generic method to get an emebedding not based only on the document id
# d2v_doc_embeddings = np.array([doc_model.infer_vector(data.tokens[i]) for i in range(data.shape[0])])


# use the static embedding generated by doc2vec model
d2v_doc_embeddings = np.array([doc_model.docvecs[i] for i in range(data.shape[0])])

In [ ]:
d2v_doc_embeddings.shape

(56, 100)

In [ ]:
d2v_doc_cos = cosine_similarity(d2v_doc_embeddings, d2v_doc_embeddings)

In [ ]:
def top_n_most_similar_document(doc_id, doc_sim, top_n=5):
    current_doc_sim = doc_sim[doc_id]
    return data[['president', 'year', 'text']].loc[(-current_doc_sim).argsort()[1:(top_n+1)]]

In [ ]:
top_n_most_similar_document(2, d2v_doc_cos)

president  year  \
43  Madison    1809   
31  Hayes      1877   
22  Adams      1797   
54  Monroe     1817   
46  Cleveland  1885   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [ ]:
ex_query = 'fellow citizien of the united states of america'
# infer the query (or a doc) vector using only the words (so good for new documents or queries)
q = doc_model.infer_vector(ex_query)

# print(cosine_similarity(d2v_doc_embeddings, q.reshape(1, -1)))
cosine_similarity(d2v_doc_embeddings, q.reshape(1, -1)).shape

(56, 1)

In [ ]:

def get_top_n_from_query(query, doc_embeddings, top_n=5):
    similarities = cosine_similarity(doc_embeddings, q.reshape(1, -1))
    # enumerate to get doc_number: similarity measure and sort accoring to the similarity
    sorted_index_list = sorted(enumerate(similarities), key=lambda item: item[1], reverse=True)
    return sorted_index_list[:top_n]

get_top_n_from_query(q, d2v_doc_embeddings)

[(0, array([0.9814495], dtype=float32)),
 (37, array([0.98106486], dtype=float32)),
 (40, array([0.9809127], dtype=float32)),
 (42, array([0.980842], dtype=float32)),
 (3, array([0.9804632], dtype=float32))]

In [ ]:
ex_query

'fellow citizien of the united states of america'

In [ ]:
print(data.iloc[3].preprocessed)

my fellow citizens  today we celebrate the mystery of american renewal  this ceremony is held in the depth of winter  but by the words we speak and the faces we show the world  we force the spring  a spring reborn in the world s oldest democracy  that brings forth the vision and courage to reinvent america  when our founders boldly declared america s independence to the world  and our purposes to the almighty  they knew that america  to endure  would have to change  not change for change sake  but change to preserve america s ideals  life  liberty  the pursuit of happiness 

though we march to the music of our time  our mission is timeless  each generation of american s must define what it means to be an american  on behalf of our nation  i salute my predecessor  president bush  for his half century of service to america and i thank the millions of men and women whose steadfastness and sacrifice triumphed over depression  fascism and communism 

today  a generation raised in the shadow